In [ ]:
import random
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# P1 (Carro): M1(3)->M2(2)->M3(2)
# P2 (Boneca): M2(4)->M1(1)->M3(2)
# P3 (Robô): M3(2)->M1(4)->M2(3)

jobs_data = {
    1: {"name": "P1 (Carro)", "route": [(1, 3), (2, 2), (3, 2)], "due": 10},
    2: {"name": "P2 (Boneca)", "route": [(2, 4), (1, 1), (3, 2)], "due": 8},
    3: {"name": "P3 (Robô)", "route": [(3, 2), (1, 4), (2, 3)], "due": 12}
}

In [ ]:
def calculate_fitness(chromosome):
    job_next_task = {1: 0, 2: 0, 3: 0}
    job_ready_time = {1: 0, 2: 0, 3: 0}
    machine_free_time = {1: 0, 2: 0, 3: 0}
    job_finish_time = {1: 0, 2: 0, 3: 0}

    for job_id in chromosome:
        task_idx = job_next_task[job_id]
        machine_id, duration = jobs_data[job_id]["route"][task_idx]
        
        # Início respeita a máquina estar livre e a ordem da rota do produto
        start_time = max(machine_free_time[machine_id], job_ready_time[job_id])
        end_time = start_time + duration
        
        # Atualiza estados
        machine_free_time[machine_id] = end_time
        job_ready_time[job_id] = end_time
        job_finish_time[job_id] = end_time
        job_next_task[job_id] += 1
        
    total_tardiness = sum(max(0, job_finish_time[j] - jobs_data[j]["due"]) for j in jobs_data)
    return total_tardiness

In [ ]:
def order_crossover(p1, p2):
    size = len(p1)
    child = [None] * size
    
    # Seleciona o trecho central
    start, end = sorted(random.sample(range(size), 2))
    child[start:end+1] = p1[start:end+1]
    
    # Cria uma lista com o que sobrou do Pai 2
    remaining_from_p2 = p2[:]
    for val in child:
        if val is not None:
            remaining_from_p2.remove(val)
            
    # Preenche os espaços vazios do filho com o que restou na ordem do Pai 2
    r_idx = 0
    for i in range(size):
        if child[i] is None:
            child[i] = remaining_from_p2[r_idx]
            r_idx += 1
            
    return child

def mutate(chromosome):
    idx1, idx2 = random.sample(range(len(chromosome)), 2)
    chromosome[idx1], chromosome[idx2] = chromosome[idx2], chromosome[idx1]

In [ ]:
pop_size = 30
generations = 50
base_chromo = [1,1,1,2,2,2,3,3,3]
population = [random.sample(base_chromo, len(base_chromo)) for _ in range(pop_size)]

for gen in range(generations):
    # Ordena por Fitness (Elitismo)
    population.sort(key=calculate_fitness)
    
    # Cria nova geração
    next_gen = population[:2]
    
    while len(next_gen) < pop_size:
        p1, p2 = random.sample(population[:10], 2) # Seleção por torneio entre os 10 melhores
        child = order_crossover(p1, p2)
        if random.random() < 0.2: # 20% de chance de mutação
            mutate(child)
        next_gen.append(child)
    
    population = next_gen

best_sol = population[0]
print(f"Melhor Solução: {best_sol}")
print(f"Menor Atraso Total: {calculate_fitness(best_sol)} min")